# mNGS Data Visualization by Farm Type

This notebook visualizes metagenomic sequencing data from Ukrainian farm wastewater samples.

**Farm Types**: Poultry vs Pig  
**Sample Types**: Native vs Enriched  
**Data**: Top 10 species per sample with read counts and relative abundance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## Load Data

In [ ]:
df = pd.read_csv('../sources/species_with_metadata.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(10)

## Data Summary by Farm Type

In [ ]:
# Samples per farm type
samples_per_type = df.groupby('Farm Type')['Barcode'].nunique()
print("Samples per Farm Type:")
print(samples_per_type)

# Total reads per farm type
reads_per_type = df.groupby(['Farm Type', 'Barcode'])['total_reads'].first().groupby('Farm Type').sum()
print("\nTotal Reads per Farm Type:")
print(reads_per_type)

## Visualization 1: Top Species by Farm Type (Bar Chart)

In [ ]:
# Aggregate read counts by species and farm type
species_by_farm = df.groupby(['Farm Type', 'species'])['read_count'].sum().reset_index()

# Get top 10 species for each farm type
top_poultry = species_by_farm[species_by_farm['Farm Type'] == 'Poultry'].nlargest(10, 'read_count')
top_pig = species_by_farm[species_by_farm['Farm Type'] == 'Pig'].nlargest(10, 'read_count')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Poultry
axes[0].barh(top_poultry['species'], top_poultry['read_count'], color='#2ecc71')
axes[0].set_xlabel('Read Count')
axes[0].set_title('Top 10 Species - Poultry Farms')
axes[0].invert_yaxis()

# Pig
axes[1].barh(top_pig['species'], top_pig['read_count'], color='#e74c3c')
axes[1].set_xlabel('Read Count')
axes[1].set_title('Top 10 Species - Pig Farms')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Visualization 2: Species Composition (Stacked Bar Chart)

In [ ]:
# Get top 8 species overall (excluding Unknown)
df_known = df[df['species'] != 'Unknown']
top_species = df_known.groupby('species')['read_count'].sum().nlargest(8).index.tolist()

# Create pivot table for stacked bar
df_top = df[df['species'].isin(top_species)].copy()
pivot = df_top.pivot_table(
    index='Barcode', 
    columns='species', 
    values='relative_abundance_pct', 
    fill_value=0
)

# Add farm type for sorting
barcode_farmtype = df[['Barcode', 'Farm Type']].drop_duplicates().set_index('Barcode')
pivot = pivot.join(barcode_farmtype).sort_values('Farm Type')
farm_types = pivot['Farm Type']
pivot = pivot.drop('Farm Type', axis=1)

# Plot
fig, ax = plt.subplots(figsize=(14, 7))
pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')

ax.set_xlabel('Sample (Barcode)')
ax.set_ylabel('Relative Abundance (%)')
ax.set_title('Top 8 Species Composition by Sample')
ax.legend(title='Species', bbox_to_anchor=(1.02, 1), loc='upper left')

# Add farm type annotation
for i, (barcode, farm) in enumerate(zip(pivot.index, farm_types)):
    color = '#2ecc71' if farm == 'Poultry' else '#e74c3c'
    ax.get_xticklabels()[i].set_color(color)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Color legend: Green = Poultry, Red = Pig")

## Visualization 3: Heatmap - Species Abundance

In [ ]:
# Create heatmap data - top 15 species
top15_species = df_known.groupby('species')['read_count'].sum().nlargest(15).index.tolist()
df_heatmap = df[df['species'].isin(top15_species)].copy()

# Pivot for heatmap
heatmap_data = df_heatmap.pivot_table(
    index='species',
    columns='Barcode',
    values='relative_abundance_pct',
    fill_value=0
)

# Sort columns by farm type
barcode_order = df[['Barcode', 'Farm Type']].drop_duplicates().sort_values('Farm Type')['Barcode'].tolist()
heatmap_data = heatmap_data[barcode_order]

# Plot
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    heatmap_data, 
    cmap='YlOrRd', 
    annot=False, 
    ax=ax,
    cbar_kws={'label': 'Relative Abundance (%)'}
)

ax.set_title('Species Abundance Heatmap (Top 15 Species)')
ax.set_xlabel('Sample (Barcode)')
ax.set_ylabel('Species')

# Color x-axis labels by farm type
farm_type_map = df[['Barcode', 'Farm Type']].drop_duplicates().set_index('Barcode')['Farm Type'].to_dict()
for label in ax.get_xticklabels():
    barcode = label.get_text()
    color = '#2ecc71' if farm_type_map.get(barcode) == 'Poultry' else '#e74c3c'
    label.set_color(color)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("X-axis color: Green = Poultry, Red = Pig")

## Visualization 4: Native vs Enriched Comparison

In [ ]:
# Calculate total reads per sample
sample_reads = df.groupby(['Barcode', 'Farm Type', 'Sample Type'])['total_reads'].first().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Box plot: Total reads by Sample Type and Farm Type
sns.boxplot(
    data=sample_reads, 
    x='Sample Type', 
    y='total_reads', 
    hue='Farm Type',
    palette={'Poultry': '#2ecc71', 'Pig': '#e74c3c'},
    ax=axes[0]
)
axes[0].set_title('Total Reads: Native vs Enriched')
axes[0].set_ylabel('Total Reads')
axes[0].set_xlabel('Sample Type')

# Species diversity (unique species count) by sample type
diversity = df.groupby(['Barcode', 'Farm Type', 'Sample Type'])['species'].nunique().reset_index()
diversity.columns = ['Barcode', 'Farm Type', 'Sample Type', 'Species Count']

sns.boxplot(
    data=diversity, 
    x='Sample Type', 
    y='Species Count', 
    hue='Farm Type',
    palette={'Poultry': '#2ecc71', 'Pig': '#e74c3c'},
    ax=axes[1]
)
axes[1].set_title('Species Diversity: Native vs Enriched')
axes[1].set_ylabel('Number of Species (Top 10)')
axes[1].set_xlabel('Sample Type')

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"\nTotal samples: {df['Barcode'].nunique()}")
print(f"Total species detected: {df['species'].nunique()}")
print(f"\nSamples by Farm Type:")
print(df.groupby('Farm Type')['Barcode'].nunique())
print(f"\nMost abundant species overall:")
print(df_known.groupby('species')['read_count'].sum().nlargest(5))